# Notebook 05 — Controles de rigor

Este notebook no cambia ninguna decisión del proyecto: **comprueba que los resultados de los notebooks 01 a 04 no dependen de errores ni de fugas de información**. Lo hacemos con cuatro controles:

1. Que `train.csv`, `test.csv` y `test_original.csv` son exactamente lo que sale de aplicar el notebook 01 al archivo ARFF original.
2. Que no hay perfiles idénticos a la vez en entrenamiento y prueba.
3. Que partir los turnos al azar no infla los resultados, aunque los turnos están en orden temporal y se parecen a sus vecinos: comparamos la validación cruzada al azar con una validación por bloques contiguos en el tiempo.
4. Que el pipeline no encuentra "señal" cuando no la hay: con las etiquetas mezcladas al azar debe rendir como el azar.

Una aclaración sobre la prueba: este notebook **no entrena ni evalúa nada con `test.csv`**. Solo compara el contenido de los archivos con lo que se obtiene al rehacer la partición desde el ARFF, igual que hace una suma de comprobación.

In [1]:
# Si corremos en Google Colab ponemos True; en el computador (VS Code), False
EN_COLAB = False

if EN_COLAB:
    # En Colab hay que instalar lo que no viene de fábrica, y se repite en cada sesión nueva
    !pip install phik shap gradio imbalanced-learn -q
    # En Colab los archivos viven en Drive, así que primero lo montamos
    from google.colab import drive
    drive.mount('/content/drive')
    RUTA_BASE = '/content/drive/MyDrive/Proyecto_Sismos/'
else:
    RUTA_BASE = './'

# Todas las rutas del notebook salen de aquí, para no tener rutas regadas por el código
RUTA_DATOS = RUTA_BASE + 'data/'
RUTA_MODELOS = RUTA_BASE + 'models/'
RUTA_RESULTADOS = RUTA_BASE + 'resultados/'
RUTA_FIGURAS = RUTA_BASE + 'figuras/'

In [2]:
import os
import joblib
import numpy as np
import pandas as pd

from scipy.io import arff
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import roc_auc_score, average_precision_score
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler

os.makedirs(RUTA_RESULTADOS, exist_ok=True)

## Rehacer la partición del notebook 01

Repetimos los pasos del notebook 01 (cargar el ARFF, quitar duplicados y columnas constantes, partir 80/20 estratificado con semilla 42 y codificar). Las funciones son copias idénticas de las del notebook 01. Si todo está bien, lo que obtengamos aquí debe coincidir con los archivos guardados.

In [3]:
def cargar_arff(ruta):
    """
    Lee el archivo .arff oficial de UCI y lo devuelve como DataFrame.
    Recibe la ruta del archivo. scipy entrega las columnas de texto como bytes (b'a'), así que aquí
    las pasamos a texto normal ('a') para poder trabajarlas igual que si vinieran de un CSV.
    """
    datos_arff, meta = arff.loadarff(ruta)
    df = pd.DataFrame(datos_arff)
    for columna in df.columns:
        # Solo las columnas de texto llegan como bytes; las numéricas ya vienen como float
        if df[columna].dtype == object:
            df[columna] = df[columna].str.decode('utf-8')
    return df


def columnas_constantes(df):
    """
    Devuelve la lista de columnas que tienen un solo valor en todas las filas.
    Recibe un DataFrame. Las buscamos porque una columna que nunca cambia no aporta información
    para separar turnos peligrosos de los que no lo son, y solo estorba en el modelo.
    """
    constantes = []
    for columna in df.columns:
        if df[columna].nunique() == 1:
            constantes.append(columna)
    return constantes


def codificar_categoricas(df, mapeos):
    """
    Cambia las letras de las columnas categóricas por números usando el diccionario `mapeos`.
    Recibe un DataFrame con las columnas originales (por ejemplo seismic = 'a' o 'b') y devuelve
    una copia con esas columnas ya numéricas. No modifica el DataFrame que recibe.
    """
    df_codificado = df.copy()
    for columna in mapeos:
        # Si la columna no está (por ejemplo, porque la quitamos en la selección de rasgos), la saltamos
        if columna in df_codificado.columns:
            # Usamos el mismo diccionario en los notebooks y en la app, así una 'b' siempre vale 1
            df_codificado[columna] = df_codificado[columna].map(mapeos[columna])
    return df_codificado


datos = cargar_arff(RUTA_DATOS + 'seismic-bumps.arff')
datos['class'] = datos['class'].astype(int)
datos = datos.drop_duplicates().reset_index(drop=True)
datos = datos.drop(columns=columnas_constantes(datos))

X = datos.drop(columns=['class'])
y = datos['class']
X_train_original, X_test_original, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

mapeos = joblib.load(RUTA_MODELOS + 'mapeos_categoricas.joblib')
X_train = codificar_categoricas(X_train_original, mapeos)
X_test = codificar_categoricas(X_test_original, mapeos)

# Los archivos esperados, con el mismo formato con que los escribió el notebook 01
esperados = {
    'train.csv': pd.concat([X_train, y_train], axis=1),
    'test.csv': pd.concat([X_test, y_test], axis=1),
    'test_original.csv': pd.concat([X_test_original, y_test], axis=1),
}
rows = []
for nombre_archivo, tabla_esperada in esperados.items():
    with open(RUTA_DATOS + nombre_archivo, encoding='utf-8', newline=None) as archivo:
        texto_guardado = archivo.read()
    texto_esperado = tabla_esperada.to_csv(index=False, lineterminator='\n')
    rows.append({'Archivo': nombre_archivo, 'Filas': len(tabla_esperada), 'Idéntico_al_rehecho': texto_guardado == texto_esperado})

tabla_integridad = pd.DataFrame(rows)
tabla_integridad

,Archivo,Filas,Idéntico_al_rehecho
0,train.csv,2062,True
1,test.csv,516,True
2,test_original.csv,516,True


**Lo que notamos.** Los tres archivos guardados (`train.csv` con 2.062 filas, `test.csv` con 516 y `test_original.csv` con 516) son idénticos, carácter por carácter, a lo que sale de rehacer el notebook 01 desde el ARFF original. O sea que la partición es reproducible, `test.csv` y `test_original.csv` corresponden a los mismos turnos y nadie los modificó después.

## Perfiles idénticos entre entrenamiento y prueba

Si un mismo perfil de mediciones aparece en entrenamiento y en prueba, el modelo podría "acertar" en prueba solo por haberlo visto. Quitamos los duplicados antes de partir justamente para evitarlo. Lo comprobamos contando los perfiles de las entradas que están en los dos conjuntos.

In [4]:
perfiles_train = X_train.drop_duplicates()
perfiles_test = X_test.drop_duplicates()
perfiles_compartidos = pd.merge(perfiles_train, perfiles_test, how='inner')
print('Perfiles distintos en entrenamiento:', len(perfiles_train))
print('Perfiles distintos en prueba:', len(perfiles_test))
print('Perfiles presentes en ambos:', len(perfiles_compartidos))

Perfiles distintos en entrenamiento: 2062
Perfiles distintos en prueba: 516
Perfiles presentes en ambos: 0


**Lo que notamos.** Hay 2.062 perfiles distintos en entrenamiento y 516 en prueba, y **ninguno aparece en los dos conjuntos**. Quitar los 6 duplicados antes de partir cumplió su propósito: el modelo no puede acertar en prueba por haber visto el mismo perfil en entrenamiento.

## ¿Infla los resultados partir los turnos al azar?

Los turnos del conjunto de datos vienen en orden temporal: las mediciones de un turno se parecen a las del turno vecino. Partir al azar deja turnos vecinos repartidos entre entrenamiento y validación, y eso podría hacer que el modelo se vea mejor de lo que es. Primero medimos qué tanto se parecen los turnos consecutivos (la autocorrelación de un turno con el siguiente, en el orden original) y después comparamos dos validaciones cruzadas con las mismas filas de entrenamiento:

- la validación al azar que usamos en todo el proyecto (10 particiones estratificadas);
- una validación por **bloques**: ordenamos las filas de entrenamiento en el orden original y cortamos 10 bloques contiguos. Cada bloque se valida con un modelo entrenado con todos los demás, así que los vecinos en el tiempo casi nunca quedan en lados distintos.

Para no depender del umbral comparamos el ROC AUC y la precisión promedio, que miden qué tan bien ordena el modelo a los turnos.

In [5]:
rows = []
for columna in ['nbumps', 'gpuls', 'genergy', 'energy']:
    rows.append({'Variable': columna, 'Autocorrelacion_con_el_turno_siguiente': datos[columna].autocorr(1)})
rows.append({'Variable': 'class', 'Autocorrelacion_con_el_turno_siguiente': datos['class'].autocorr(1)})
tabla_autocorrelacion = pd.DataFrame(rows)
tabla_autocorrelacion.round(3)

,Variable,Autocorrelacion_con_el_turno_siguiente
0,nbumps,0.518
1,gpuls,0.730
2,genergy,0.761
3,energy,0.057
4,class,0.074


In [6]:
decisiones = joblib.load(RUTA_MODELOS + 'decisiones_preprocesamiento.joblib')
rasgos = decisiones['rasgos']


def crear_pipeline_svm_lineal():
    """Arma el pipeline del modelo final (escalado, submuestreo y SVM lineal), sin calibrar, para puntuar turnos."""
    return Pipeline([
        ('escalador', MinMaxScaler()),
        ('balanceo', RandomUnderSampler(random_state=42)),
        ('modelo', LinearSVC(C=0.1, max_iter=10000, random_state=42)),
    ])


def crear_pipeline_logistica():
    """Arma un pipeline de regresión logística balanceada, como referencia más simple que el modelo final."""
    return Pipeline([
        ('escalador', MinMaxScaler()),
        ('modelo', LogisticRegression(C=1, max_iter=2000, class_weight='balanced')),
    ])


def puntuar_con_particiones(crear_pipeline, X, y, particiones):
    """
    Entrena una copia del pipeline en cada partición y puntúa la parte de validación.
    Devuelve un puntaje por fila, siempre calculado por un modelo que no vio esa fila.
    """
    puntajes = np.zeros(len(y))
    for indices_train, indices_validacion in particiones:
        pipeline = crear_pipeline()
        pipeline.fit(X.iloc[indices_train], y.iloc[indices_train])
        if hasattr(pipeline, 'decision_function'):
            puntajes[indices_validacion] = pipeline.decision_function(X.iloc[indices_validacion])
        else:
            puntajes[indices_validacion] = pipeline.predict_proba(X.iloc[indices_validacion])[:, 1]
    return puntajes


# Las filas de entrenamiento, en el orden en que aparecían en el archivo original
orden_original = np.argsort(X_train.index.to_numpy())
X_ordenado = X_train[rasgos].iloc[orden_original].reset_index(drop=True)
y_ordenado = y_train.iloc[orden_original].reset_index(drop=True)

particiones_azar = list(StratifiedKFold(n_splits=10, shuffle=True, random_state=42).split(X_ordenado, y_ordenado))
particiones_bloques = []
for bloque in np.array_split(np.arange(len(X_ordenado)), 10):
    particiones_bloques.append((np.setdiff1d(np.arange(len(X_ordenado)), bloque), bloque))

rows = []
for nombre_modelo, crear_pipeline in [('SVM lineal + submuestreo', crear_pipeline_svm_lineal),
                                       ('Regresión logística balanceada', crear_pipeline_logistica)]:
    for nombre_validacion, particiones in [('Al azar', particiones_azar), ('Por bloques en el tiempo', particiones_bloques)]:
        puntajes = puntuar_con_particiones(crear_pipeline, X_ordenado, y_ordenado, particiones)
        rows.append({'Modelo': nombre_modelo, 'Validacion': nombre_validacion,
                     'ROC_AUC': roc_auc_score(y_ordenado, puntajes),
                     'Precision_promedio': average_precision_score(y_ordenado, puntajes)})

tabla_orden = pd.DataFrame(rows)
tabla_orden.round(3)

,Modelo,Validacion,ROC_AUC,Precision_promedio
0,SVM lineal + submuestreo,Al azar,0.774,0.208
1,SVM lineal + submuestreo,Por bloques en el tiempo,0.778,0.195
2,Regresión logística balanceada,Al azar,0.774,0.210
3,Regresión logística balanceada,Por bloques en el tiempo,0.779,0.200


**Lo que notamos.** Los turnos consecutivos sí se parecen: la autocorrelación con el turno siguiente es de 0,73 para `gpuls`, 0,76 para `genergy` y 0,52 para `nbumps`, así que los datos están en orden temporal. La clase, en cambio, casi no se repite de un turno al siguiente (0,074), y `energy` tampoco (0,057).

Pese a esa dependencia, la validación por bloques en el tiempo no empeora los resultados: con la SVM lineal el ROC AUC es 0,778 por bloques frente a 0,774 al azar, y la precisión promedio 0,195 frente a 0,208; con la regresión logística, 0,779 frente a 0,774 y 0,200 frente a 0,210. La diferencia en precisión promedio es de un punto, dentro del ruido de tener solo 136 turnos peligrosos. Es decir, **partir al azar no infla los resultados** de forma apreciable. Lo que sí sigue siendo cierto es que, sin fechas, no podemos probar una validación hacia el futuro de verdad: eso queda como limitación.

## Etiquetas mezcladas: el pipeline no debe encontrar señal donde no la hay

La prueba de las etiquetas permutadas es la comprobación más sencilla de que un pipeline no filtra información: mezclamos al azar la clase de los turnos de entrenamiento (así deja de tener relación con las mediciones) y repetimos la validación cruzada. Si hubiera una fuga, el ROC AUC seguiría muy por encima de 0,5. Si no, debe quedar alrededor de 0,5 y la precisión promedio alrededor de la proporción de turnos peligrosos.

In [7]:
y_mezclado = pd.Series(np.random.RandomState(42).permutation(y_ordenado.to_numpy()))
puntajes_mezclados = puntuar_con_particiones(crear_pipeline_svm_lineal, X_ordenado, y_mezclado, particiones_azar)

tabla_permutadas = pd.DataFrame([{
    'ROC_AUC_con_etiquetas_mezcladas': roc_auc_score(y_mezclado, puntajes_mezclados),
    'Precision_promedio_con_etiquetas_mezcladas': average_precision_score(y_mezclado, puntajes_mezclados),
    'Proporcion_de_turnos_peligrosos': y_mezclado.mean(),
}])
tabla_permutadas.round(3)

,ROC_AUC_con_etiquetas_mezcladas,Precision_promedio_con_etiquetas_mezcladas,Proporcion_de_turnos_peligrosos
0,0.473,0.06,0.066


**Lo que notamos.** Con las etiquetas mezcladas el ROC AUC es 0,473 y la precisión promedio 0,060, casi igual a la proporción de turnos peligrosos (0,066). El pipeline completo (escalado, submuestreo y modelo, con validación cruzada) no encuentra señal cuando no la hay, que es lo que esperábamos: no hay fuga de información en su mecánica.

In [8]:
tabla_integridad.to_csv(RUTA_RESULTADOS + '05_integridad_particion.csv', index=False)
tabla_orden.to_csv(RUTA_RESULTADOS + '05_validacion_azar_vs_bloques.csv', index=False)
tabla_permutadas.to_csv(RUTA_RESULTADOS + '05_etiquetas_permutadas.csv', index=False)
tabla_autocorrelacion.to_csv(RUTA_RESULTADOS + '05_autocorrelacion.csv', index=False)
print('Tablas guardadas en', RUTA_RESULTADOS)

Tablas guardadas en ./resultados/


## Conclusiones

- **Partición.** Los archivos de datos son reproducibles desde el ARFF y no hay perfiles compartidos entre entrenamiento y prueba.
- **Orden temporal.** Los turnos vecinos se parecen (autocorrelación de hasta 0,76 en `genergy`), pero validar por bloques contiguos da el mismo ROC AUC que validar al azar (0,778 contra 0,774), así que la partición al azar no infla los resultados.
- **Mecánica del pipeline.** Con etiquetas mezcladas queda en el nivel del azar (ROC AUC de 0,473), sin fuga.
- **Lo que estos controles no pueden resolver.** No comprueban que el resultado de prueba sea "virgen": ese punto (la prueba se consultó una vez con una primera versión del notebook 04) está explicado en ese notebook y en el informe, y por eso el notebook 04 incluye también una estimación anidada que no usa la prueba. Tampoco prueban generalización a otra mina ni hacia el futuro, porque no hay fechas ni datos de otras minas.
- **Archivos generados.** `resultados/05_integridad_particion.csv`, `resultados/05_validacion_azar_vs_bloques.csv`, `resultados/05_etiquetas_permutadas.csv` y `resultados/05_autocorrelacion.csv`.